# Example notebook using the IGVF Catalog API to perform comparative analysis between HepG2 lentiMPRA data ([IGVFFI4134MFLL](https://data.igvf.org/tabular-files/IGVFFI4134MFLL/)) and eQTLs from the EBI eQTL Catalogue

In [2]:
import requests
import json
from pprint import pprint as pp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
def paginated(url, page_size=25, max_results=1000, verbose=False):
    ''' This is a utility function to page results from the Catalog API
        use page_size (default = 25 records) and max_results (default=1000 records)
        One query is made per page, up to max_results.
    '''

    page = 0
    found = 'start'
    data = []
    total_found = 0
    request_url = url + "&page=%s&limit=%s" % (page, page_size)
    while(found and len(data) <= max_results):
        if found == 'start':
            found = 0
        res = requests.get(url+ "&page=%s&limit=%s" % (page, page_size)  )
        if res.status_code == 200:
            found = len(res.json())
            total_found = total_found + found
            if verbose:
                print("url: %s page: %s found: %s total: %s" % (request_url, page, found, total_found))
            # last page should be empty
            for doc in res.json():
                data.append(doc)
            page=page+1
        else:
            print(url+ "&page=%s&limit=%s" % (page, page_size)  )
            print(res.text)
            break
    return data

In [6]:
api_host = 'https://catalog-api-dev.demo.igvf.org'

## MPRA

### Retrieve data from [IGVFFI4134MFLL](https://data.igvf.org/tabular-files/IGVFFI4134MFLL/), the `reporter genomic variant effects` file under analysis set [IGVFDS6149YGOL](https://data.igvf.org/analysis-sets/IGVFDS6149YGOL/)

In [45]:
file_id = 'IGVFFI4134MFLL'
url = "%s/api/variants/biosamples?files_fileset=%s&verbose=true" % (api_host, file_id)
print(url)

byfile = paginated(url, 500, 150_000)
flattened_byfile = []
for rec in byfile:
    flattened_rec = {}
    for k, v in rec.items():
        if isinstance(v, dict) and k != 'variant':
            for kk, vv in v.items():
                flattened_rec[f"{k}_{kk}"] = vv
        elif isinstance(v, list):
            for i, vv in enumerate(v):
                new_rec = flattened_rec.copy()
            for kkk,vvv in vv.items():
                new_rec[f"{kkk}"] = vvv
            flattened_byfile.append(new_rec)
        else:
            flattened_rec[k] = v
    flattened_byfile.append(flattened_rec)
variantMPRA_df = pd.DataFrame(flattened_byfile)
variantMPRA_df

https://catalog-api-dev.demo.igvf.org/api/variants/biosamples?files_fileset=IGVFFI4134MFLL&verbose=true


,variant,biosample_uri,biosample_term_id,biosample_name,biosample_synonyms,biosample_description,biosample_source,biosample_subontology,biosample_source_url,genomic_element__id,...,significant,neg_log10_pvalue,neg_log10_pvalue_adj,label,method,class,source,source_url,name,files_filesets
0,"{'organism': 'Homo sapiens', '_id': 'NC_000001...",http://www.ebi.ac.uk/efo/EFO_0001187,EFO_0001187,hepg2,"[Hep-G2, HepG2 cell]",human hepatocellular carcinoma (HCC) cells (p5...,EFO,None,https://www.ebi.ac.uk/efo/,MPRA_chr1_1000079_1000279_GRCh38_plus_IGVFFI73...,...,False,0.1185,0.0252,variant effect on regulatory element activity,MPRA,observed data,IGVF,https://data.igvf.org/tabular-files/IGVFFI4134...,modulates regulatory activity of,files_filesets/IGVFFI4134MFLL
1,"{'organism': 'Homo sapiens', '_id': 'NC_000001...",http://www.ebi.ac.uk/efo/EFO_0001187,EFO_0001187,hepg2,"[Hep-G2, HepG2 cell]",human hepatocellular carcinoma (HCC) cells (p5...,EFO,None,https://www.ebi.ac.uk/efo/,MPRA_chr1_1000079_1000279_GRCh38_plus_IGVFFI73...,...,False,1.9360,0.8213,variant effect on regulatory element activity,MPRA,observed data,IGVF,https://data.igvf.org/tabular-files/IGVFFI4134...,modulates regulatory activity of,files_filesets/IGVFFI4134MFLL
2,"{'organism': 'Homo sapiens', '_id': 'NC_000001...",http://www.ebi.ac.uk/efo/EFO_0001187,EFO_0001187,hepg2,"[Hep-G2, HepG2 cell]",human hepatocellular carcinoma (HCC) cells (p5...,EFO,None,https://www.ebi.ac.uk/efo/,MPRA_chr1_100029009_100029209_GRCh38_plus_IGVF...,...,False,0.8310,0.2453,variant effect on regulatory element activity,MPRA,observed data,IGVF,https://data.igvf.org/tabular-files/IGVFFI4134...,modulates regulatory activity of,files_filesets/IGVFFI4134MFLL
3,"{'organism': 'Homo sapiens', '_id': 'NC_000001...",http://www.ebi.ac.uk/efo/EFO_0001187,EFO_0001187,hepg2,"[Hep-G2, HepG2 cell]",human hepatocellular carcinoma (HCC) cells (p5...,EFO,None,https://www.ebi.ac.uk/efo/,MPRA_chr1_100036889_100037089_GRCh38_plus_IGVF...,...,False,0.3026,0.0712,variant effect on regulatory element activity,MPRA,observed data,IGVF,https://data.igvf.org/tabular-files/IGVFFI4134...,modulates regulatory activity of,files_filesets/IGVFFI4134MFLL
4,"{'organism': 'Homo sapiens', '_id': 'NC_000001...",http://www.ebi.ac.uk/efo/EFO_0001187,EFO_0001187,hepg2,"[Hep-G2, HepG2 cell]",human hepatocellular carcinoma (HCC) cells (p5...,EFO,None,https://www.ebi.ac.uk/efo/,MPRA_chr1_100036889_100037089_GRCh38_plus_IGVF...,...,False,0.2156,0.0478,variant effect on regulatory element activity,MPRA,observed data,IGVF,https://data.igvf.org/tabular-files/IGVFFI4134...,modulates regulatory activity of,files_filesets/IGVFFI4134MFLL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147527,"{'organism': 'Homo sapiens', '_id': 'NC_000024...",http://www.ebi.ac.uk/efo/EFO_0001187,EFO_0001187,hepg2,"[Hep-G2, HepG2 cell]",human hepatocellular carcinoma (HCC) cells (p5...,EFO,None,https://www.ebi.ac.uk/efo/,MPRA_chrY_7874162_7874362_GRCh38_minus_IGVFFI7...,...,False,0.5974,0.1621,variant effect on regulatory element activity,MPRA,observed data,IGVF,https://data.igvf.org/tabular-files/IGVFFI4134...,modulates regulatory activity of,files_filesets/IGVFFI4134MFLL
147528,"{'organism': 'Homo sapiens', '_id': 'NC_000024...",http://www.ebi.ac.uk/efo/EFO_0001187,EFO_0001187,hepg2,"[Hep-G2, HepG2 cell]",human hepatocellular carcinoma (HCC) cells (p5...,EFO,None,https://www.ebi.ac.uk/efo/,MPRA_chrY_7874162_7874362_GRCh38_minus_IGVFFI7...,...,False,0.7380,0.2106,variant effect on regulatory element activity,MPRA,observed data,IGVF,https://data.igvf.org/tabular-files/IGVFFI4134...,modulates regulatory activity of,files_filesets/IGVFFI4134MFLL
147529,"{'organism': 'Homo sapiens', '_id': 'NC_000024...",http://www.ebi.ac.uk/efo/EFO_0001187,EFO_0001187,hepg2,"[Hep-G2, HepG2 cell]",human hepatocellular carcinoma (HCC) cells (p5...,EFO,None,https://www.ebi.ac.uk/efo/,MPRA_chrY_7874162_7874362_GRCh38_minus_IGVFFI7...,...,False,1.2217

## eQTL

In [ ]:
eqtls = []
for variant in variantMPRA_df['variant']:
    url = "%s/api/qtls?spdi=%s&biological_context=liver&method=eQTL&source=EBI&organism=Homo sapiens" % (api_host, variant['spdi'])
    res = requests.get(url)
    if res.status_code == 200 and res.json():
        eqtls.extend(res.json())